# Comparing London and Paris

Runs after `Airbnb_Project_Guided.ipynb` has been run for both cities. Two questions: do the predictors behave the same in both cities, and which findings of Biswas et al. (2020) and Sengupta et al. (2021) hold?

In [1]:
# Setup: both cities' model data from the analysis notebook (run it for London and Paris first)
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from models import CORE, EXPECTED, replication

TABLES = "../outputs/tables"
data = {c: pd.read_csv(f"../data/{c}/{c}_model_data.csv.gz") for c in ["London", "Paris"]}
print({c: len(d) for c, d in data.items()})

{'London': 50000, 'Paris': 49999}


## Pooled model with city interactions
Model A fitted on both cities, with a `predictor × paris` term for every predictor. A significant interaction means the effect differs between the cities.

In [2]:
# Pooled Model A with a city interaction on every predictor
# Predictors are standardised within each city, so effects are per SD of the local market
Pool = pd.concat([d[["host_id", "number_of_reviews"] + CORE].assign(paris=int(c == "Paris")) for c, d in data.items()],
                 ignore_index=True)
pooled = smf.negativebinomial(f"number_of_reviews ~ ({' + '.join(CORE)}) * paris", Pool).fit(
    method="bfgs", maxiter=5000, disp=0, cov_type="cluster", cov_kwds={"groups": Pool["host_id"]})
assert pooled.mle_retvals["converged"], "Pooled model did not converge"

b, inter = pooled.params, [f"{v}:paris" for v in CORE]
cities = pd.DataFrame({
    "London IRR": np.exp(b[CORE]).values,
    "Paris IRR": np.exp(b[CORE].values + b[inter].values),
    "interaction p": pooled.pvalues[inter].values,
}, index=CORE)
cities["differs at 5%"] = cities["interaction p"] < .05
print(f"Pooled listings: {len(Pool):,}")
print(cities.round(3).to_string())
cities.to_csv(f"{TABLES}/pooled_interactions.csv")

Pooled listings: 99,999
                     London IRR  Paris IRR  interaction p  differs at 5%
superhost                 2.144      1.968          0.064          False
host_years_z              1.618      1.425          0.000           True
log_host_listings_z       1.041      0.916          0.078          False
accommodates_z            0.945      1.007          0.060          False
n_amenities_z             1.760      1.627          0.002           True
entire_home               0.750      0.513          0.000           True
log_price_z               0.767      0.915          0.000           True
price_missing             0.577      0.524          0.109          False


## Replication scorecard
✅ significant in the direction the papers report, ❌ significant in the opposite direction, n.s. not significant at 5%. Directions: Biswas et al. (2020) Tables 4 to 7; Sengupta et al. (2021) Tables 5 to 8. Neither paper includes Paris; Biswas reports London as its own cluster (Cluster 4); Sengupta's nearest comparison is its Europe model.

In [3]:
# Replication scorecard: our estimates against the direction reported by Biswas et al. (2020) and Sengupta et al. (2021)
# Core variables from Model A, text variables from Model A+, guest-side variables from Model B
def verdict(t, v, sign):
    if v not in t.index:
        return "not estimated"
    found = "+" if t.loc[v, "IRR"] > 1 else "-"
    if t.loc[v, "p"] >= .05:
        return f"n.s. ({t.loc[v, 'IRR']:.2f})"
    return f"{'✅' if found == sign else '❌'} {t.loc[v, 'IRR']:.2f}"

card = pd.DataFrame({"expected": {v: s for v, (s, _) in EXPECTED.items()}, "source": {v: src for v, (_, src) in EXPECTED.items()}})
for c, m in data.items():
    estimates = replication(m)
    card[c] = [verdict(estimates, v, s) for v, (s, _) in EXPECTED.items()]

untestable = pd.DataFrame({"expected": ["faster +", "+"], "source": ["Sengupta", "Both"], "London": "⚠️ empty in file", "Paris": "⚠️ empty in file"},
                          index=["host_response_time", "instant_bookable"])
card = pd.concat([card, untestable])
print(card.to_string())
card.to_csv(f"{TABLES}/scorecard.csv")

                               expected    source            London             Paris
superhost                             +      Both            ✅ 2.14            ✅ 1.97
host_years_z                          +      Both            ✅ 1.62            ✅ 1.43
log_host_listings_z                   -  Sengupta       n.s. (1.04)       n.s. (0.92)
accommodates_z                        -  Sengupta            ✅ 0.95       n.s. (1.01)
n_amenities_z                         +      Both            ✅ 1.76            ✅ 1.63
log_price_z                           -    Biswas            ✅ 0.77            ✅ 0.91
has_profile_pic                       +      Both            ✅ 1.93            ✅ 1.51
identity_verified                     +    Biswas       n.s. (0.87)       n.s. (0.83)
log_desc_words_z                      +      Both            ✅ 1.25            ✅ 1.11
lix_z                                 +    Biswas       n.s. (1.00)       n.s. (1.00)
promo_per_100_z                       +    Biswas     